# 04 - Data Preprocessing

## Objective

The objective of this notebook is to prepare the processed UAV network intrusion dataset for Machine Learning and Deep Learning model development.

The preprocessing pipeline includes:

- Removing non-informative features.
- Converting data into a model-compatible format.
- Encoding the target labels.
- Splitting the dataset into training and testing sets.
- Scaling numerical features where required.
- Saving the processed datasets for subsequent model development.

The resulting dataset will serve as the input for the Machine Learning and Deep Learning models developed in the following notebooks.


In [3]:
# Data Manipulation
import pandas as pd
import numpy as np

# Visualization (for verification if required)
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# Display Settings
pd.set_option("display.max_columns", None)

In [4]:
# Load processed dataset
df = pd.read_csv("../data/interim/network_dataset.csv")

# Display first five rows
df.head()

,timestamp_c,frame.number,frame.len,frame.protocols,wlan.duration,wlan.ra,wlan.ta,wlan.da,wlan.sa,wlan.bssid,wlan.frag,wlan.seq,llc.type,ip.hdr_len,ip.len,ip.id,ip.flags,ip.ttl,ip.proto,ip.src,ip.dst,tcp.srcport,tcp.dstport,tcp.seq_raw,tcp.ack_raw,tcp.hdr_len,tcp.flags,tcp.window_size,tcp.options,udp.srcport,udp.dstport,udp.length,data.data,data.len,wlan.fc.type,wlan.fc.subtype,time_since_last_packet,class
0,28105.97520,60,24,0,0,1,1,1,1,0,0,24,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,4,0.000000,benign
1,28105.97550,61,24,0,0,1,1,1,1,0,0,26,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,4,0.000298,benign
2,28107.09931,75,104,0,0,4,1,4,1,0,0,49,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8,1.123815,benign
3,28114.78570,122,86,2,44,1,0,1,0,0,0,14,2048,20,52,39340,64,128,6,2,4,62951,53,723862127,0,32,2,64240,1,0,0,0,0,0,2,8,7.686387,benign
4,28114.88188,124,26,0,60,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,12,0.096183,benign


In [5]:
print(f"Dataset Shape: {df.shape}")

Dataset Shape: (33102, 38)


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 33102 entries, 0 to 33101
Data columns (total 38 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   timestamp_c             33102 non-null  float64
 1   frame.number            33102 non-null  int64  
 2   frame.len               33102 non-null  int64  
 3   frame.protocols         33102 non-null  int64  
 4   wlan.duration           33102 non-null  int64  
 5   wlan.ra                 33102 non-null  int64  
 6   wlan.ta                 33102 non-null  int64  
 7   wlan.da                 33102 non-null  int64  
 8   wlan.sa                 33102 non-null  int64  
 9   wlan.bssid              33102 non-null  int64  
 10  wlan.frag               33102 non-null  int64  
 11  wlan.seq                33102 non-null  int64  
 12  llc.type                33102 non-null  int64  
 13  ip.hdr_len              33102 non-null  int64  
 14  ip.len                  33102 non-null  int64  
 

## Removing Constant Features

Features with zero variance contain the same value across all observations and therefore do not contribute to distinguishing between different classes.

Such features do not provide useful information for Machine Learning or Deep Learning models and are removed to simplify the dataset while preserving its predictive capability.

In [7]:
# Identify constant features
constant_features = [
    col for col in df.columns
    if df[col].nunique() == 1
]
# Remove constant features
df.drop(columns=constant_features, inplace=True)

print(f"Updated Dataset Shape: {df.shape}")

Updated Dataset Shape: (33102, 35)


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 33102 entries, 0 to 33101
Data columns (total 35 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   timestamp_c             33102 non-null  float64
 1   frame.number            33102 non-null  int64  
 2   frame.len               33102 non-null  int64  
 3   frame.protocols         33102 non-null  int64  
 4   wlan.duration           33102 non-null  int64  
 5   wlan.ra                 33102 non-null  int64  
 6   wlan.ta                 33102 non-null  int64  
 7   wlan.da                 33102 non-null  int64  
 8   wlan.sa                 33102 non-null  int64  
 9   wlan.seq                33102 non-null  int64  
 10  llc.type                33102 non-null  int64  
 11  ip.hdr_len              33102 non-null  int64  
 12  ip.len                  33102 non-null  int64  
 13  ip.id                   33102 non-null  int64  
 14  ip.flags                33102 non-null  int64  
 

In [9]:
# Features to remove
irrelevant_features = [
    "frame.number",
    "timestamp_c"
]

# Drop the features
df.drop(columns=irrelevant_features, inplace=True)

print("Removed Features:")
print(irrelevant_features)

print(f"\nUpdated Dataset Shape: {df.shape}")

Removed Features:
['frame.number', 'timestamp_c']

Updated Dataset Shape: (33102, 33)


## Observation

Two metadata-related features (`frame.number` and `timestamp_c`) were removed from the dataset.

These features describe the packet capture process rather than the characteristics of network traffic itself. Removing them helps prevent the model from learning dataset-specific ordering or timing information while preserving meaningful temporal behavior through the `time_since_last_packet` feature.

In [10]:
# Keep protocol sequence/identifier features (wlan.seq, ip.id, tcp.seq_raw);
# evaluate their usefulness later during feature engineering.

# Target Label Encoding

The target variable contains categorical attack labels represented as text.

Since Machine Learning algorithms require numerical target values, the class labels are converted into integer representations using **LabelEncoder**.

The label mapping is displayed and recorded to ensure consistency during model training and evaluation.

In [11]:
from sklearn.preprocessing import LabelEncoder

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Encode target labels
df["class"] = label_encoder.fit_transform(df["class"])

In [12]:
# Display label mapping
label_mapping = dict(zip(label_encoder.classes_,
                         label_encoder.transform(label_encoder.classes_)))

print("Label Encoding Mapping:")
print(label_mapping)

Label Encoding Mapping:
{'DoS attack': np.int64(0), 'Replay': np.int64(1), 'benign': np.int64(2)}


In [13]:
df.head()

,frame.len,frame.protocols,wlan.duration,wlan.ra,wlan.ta,wlan.da,wlan.sa,wlan.seq,llc.type,ip.hdr_len,ip.len,ip.id,ip.flags,ip.ttl,ip.proto,ip.src,ip.dst,tcp.srcport,tcp.dstport,tcp.seq_raw,tcp.hdr_len,tcp.flags,tcp.window_size,tcp.options,udp.srcport,udp.dstport,udp.length,data.data,data.len,wlan.fc.type,wlan.fc.subtype,time_since_last_packet,class
0,24,0,0,1,1,1,1,24,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,4,0.000000,2
1,24,0,0,1,1,1,1,26,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,4,0.000298,2
2,104,0,0,4,1,4,1,49,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8,1.123815,2
3,86,2,44,1,0,1,0,14,2048,20,52,39340,64,128,6,2,4,62951,53,723862127,32,2,64240,1,0,0,0,0,0,2,8,7.686387,2
4,26,0,60,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,12,0.096183,2


## Feature–Target Separation

The dataset is divided into:

- **Feature Matrix (X):** Contains all input variables used for model training.
- **Target Vector (y):** Contains the encoded attack labels.

Separating the features and target variable is a standard preprocessing step before model training and evaluation.

In [14]:
# Separate input features and target label

X = df.drop(columns=["class"])
y = df["class"]

print(f"Feature Matrix Shape : {X.shape}")
print(f"Target Vector Shape  : {y.shape}")

Feature Matrix Shape : (33102, 32)
Target Vector Shape  : (33102,)


In [15]:
X.head()

,frame.len,frame.protocols,wlan.duration,wlan.ra,wlan.ta,wlan.da,wlan.sa,wlan.seq,llc.type,ip.hdr_len,ip.len,ip.id,ip.flags,ip.ttl,ip.proto,ip.src,ip.dst,tcp.srcport,tcp.dstport,tcp.seq_raw,tcp.hdr_len,tcp.flags,tcp.window_size,tcp.options,udp.srcport,udp.dstport,udp.length,data.data,data.len,wlan.fc.type,wlan.fc.subtype,time_since_last_packet
0,24,0,0,1,1,1,1,24,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,4,0.000000
1,24,0,0,1,1,1,1,26,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,4,0.000298
2,104,0,0,4,1,4,1,49,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8,1.123815
3,86,2,44,1,0,1,0,14,2048,20,52,39340,64,128,6,2,4,62951,53,723862127,32,2,64240,1,0,0,0,0,0,2,8,7.686387
4,26,0,60,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,12,0.096183


In [16]:
y.head()

0    2
1    2
2    2
3    2
4    2
Name: class, dtype: int64

## Observation

The dataset has been successfully separated into input features and target labels.

- The feature matrix contains **32 predictive features**.
- The target vector contains the encoded attack classes.

This separation prepares the dataset for train-test splitting and subsequent model development.

The dataset is divided into separate training and testing sets.

- The **training set** is used to learn the relationship between network features and attack classes.
- The **testing set** is kept completely unseen during training and is used to evaluate model performance.

A **stratified split** is used to preserve the original class distribution in both subsets, ensuring a fair evaluation.

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [18]:
print("Training Features :", X_train.shape)
print("Testing Features  :", X_test.shape)
print("Training Labels   :", y_train.shape)
print("Testing Labels    :", y_test.shape)

Training Features : (26481, 32)
Testing Features  : (6621, 32)
Training Labels   : (26481,)
Testing Labels    : (6621,)


In [19]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns
)

X_test = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns
)

In [20]:
print("Training Class Distribution")
print(y_train.value_counts(normalize=True))

print("\nTesting Class Distribution")
print(y_test.value_counts(normalize=True))

Training Class Distribution
class
1    0.362675
0    0.352592
2    0.284732
Name: proportion, dtype: float64

Testing Class Distribution
class
1    0.362785
0    0.352515
2    0.284700
Name: proportion, dtype: float64


Since our dataset contains multiple attack classes, a random split could accidentally produce unequal class distributions in the training and testing sets. Using stratification preserves the original class proportions, leading to a more reliable and unbiased evaluation of the intrusion detection models.

In [21]:
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)

In [22]:
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)

y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

In [23]:
import os

os.listdir("../data/processed")

['X_test.csv', 'X_train.csv', 'y_test.csv', 'y_train.csv']

### Observation

The processed dataset has been successfully saved for downstream machine learning experiments. Subsequent notebooks will directly load these files, ensuring a consistent preprocessing pipeline across all models.